# 19.2 Interpretability: LIME & saliency maps

LIME explains a complicated prediction by fitting a small local surrogate, while saliency reads the local gradient directly.

This notebook replaces the known-bad generic guarded-score template with real weighted least squares, finite-difference saliency, and local faithfulness checks.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

Saliency uses the first-order approximation $f(x+\Delta)pprox f(x)+
abla f(x)^	op\Delta$. LIME fits a weighted local linear model around the same point, so the coefficients should recover the local components when the model is linear.

In [ ]:

def local_explain(x, model, kernel_width=1.0, n_samples=256, seed=0):
    rng = np.random.default_rng(seed)
    perturb = rng.normal(0.0, 0.35, size=(n_samples, len(x)))
    Z = x + perturb
    distances = np.linalg.norm(Z - x, axis=1)
    weights = np.exp(-(distances ** 2) / (kernel_width ** 2))
    y = np.array([model(row) for row in Z])
    design = np.column_stack([np.ones(n_samples), Z - x])
    weighted_design = design * np.sqrt(weights[:, None])
    weighted_y = y * np.sqrt(weights)
    beta = np.linalg.lstsq(weighted_design, weighted_y, rcond=None)[0]
    return beta[0], beta[1:], Z, weights, y


def toy_lime_model(z):
    weights = np.array([0.8, 0.2, -0.1])
    return float(weights @ z)

x_toy = np.zeros(3)
delta_toy = np.ones(3)
intercept_toy, coef_toy, Z_toy, w_toy, y_toy = local_explain(x_toy, toy_lime_model, kernel_width=0.75)
grad_toy = finite_gradient(toy_lime_model, x_toy)
components_toy = grad_toy * delta_toy
total_toy = float(components_toy.sum())
abs_toy = float(np.abs(components_toy).sum())
share_toy = float(np.max(np.abs(components_toy)) / abs_toy)
guarded_toy = total_toy + 0.5 * abs_toy
contrast_toy = total_toy - components_toy[2]

assert np.allclose(np.round(coef_toy, 3), [0.8, 0.2, -0.1])
assert np.allclose(components_toy, [0.8, 0.2, -0.1])
assert np.isclose(total_toy, 0.9)
assert np.isclose(abs_toy, 1.1)
assert np.isclose(round(share_toy, 3), 0.727)
assert np.isclose(guarded_toy, 1.45)
assert np.isclose(contrast_toy - total_toy, 0.1)

print("LIME coefficients", np.round(coef_toy, 3))
print("saliency components", components_toy)
print("top share", round(share_toy, 3))


For the ladder we explain one held-out point. The same function perturbs the standardized point, fits a weighted Ridge surrogate, and compares the surrogate $R^2$ with saliency deletion drop.

In [ ]:

def lime_for_classifier(model, x, kernel_width=1.0, n_samples=384, seed=0):
    class_index = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))

    def local_model(z):
        return float(score_for_class(model, z.reshape(1, -1), class_index=class_index)[0])

    intercept, coef, Z, weights, y = local_explain(x, local_model, kernel_width=kernel_width, n_samples=n_samples, seed=seed)
    centered = Z - x
    preds = intercept + centered @ coef
    local_r2 = r2_score(y, preds, sample_weight=weights)
    grad = finite_gradient(local_model, x)
    return coef, grad, local_r2, Z, y, preds


def saliency_deletion_drop(model, x, grad):
    class_index = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
    original = score_for_class(model, x.reshape(1, -1), class_index=class_index)[0]
    mask = top_k_mask(grad, max(1, len(grad) // 5))
    edited = x.copy()
    edited[mask] = 0.0
    changed = score_for_class(model, edited.reshape(1, -1), class_index=class_index)[0]
    return float(original - changed)


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is local faithfulness: weighted surrogate $R^2$, with saliency deletion drop retained as a second audit number.

In [ ]:

lime_rows = []
lime_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    width = 0.75 + 0.15 * rung
    coef, grad, local_r2, Z, y_local, preds = lime_for_classifier(model, x_te[0], kernel_width=width, seed=rung)
    deletion_drop = saliency_deletion_drop(model, x_te[0], grad)
    lime_rows.append({"rung": rung, "name": name, "kernel_width": width, "local_r2": local_r2, "saliency_drop": deletion_drop})
    lime_artifacts.append((name, coef, grad))

lime_table = pd.DataFrame(lime_rows)
print(lime_table.to_string(index=False))


## Results visualization

The panels compare LIME coefficients with saliency gradients. The summary curve tracks local surrogate faithfulness as the ladder becomes more stressed.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], lime_artifacts):
    name, coef, grad = artifact
    keep = min(10, len(coef))
    ax.plot(np.arange(keep), coef[:keep], marker="o", label="LIME")
    ax.plot(np.arange(keep), grad[:keep], marker="x", label="saliency")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(name[:26])
    ax.legend(fontsize=8)

axes[5].plot(lime_table["rung"], lime_table["local_r2"], marker="o")
axes[5].set_title("faithfulness vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("weighted local R2")
axes[5].set_ylim(0.0, 1.05)
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: a huge kernel makes the surrogate global. The fix is to tune the kernel width and check coefficient stability across perturbation counts.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
wide_coef, wide_grad, wide_r2, Z, y_local, preds = lime_for_classifier(model, x_te[0], kernel_width=10.0, seed=5)
small_coef, small_grad, small_r2, Z, y_local, preds = lime_for_classifier(model, x_te[0], kernel_width=1.0, seed=5)
stable_coef, stable_grad, stable_r2, Z, y_local, preds = lime_for_classifier(model, x_te[0], kernel_width=1.0, n_samples=768, seed=5)
stability = float(np.linalg.norm(small_coef - stable_coef) / (np.linalg.norm(stable_coef) + 1e-9))

print("huge-kernel R2", wide_r2)
print("tuned-kernel R2", small_r2)
print("perturbation-count relative coefficient change", stability)


## Evaluate it + Practice

- Compare the reported weighted local surrogate R2 with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: use a huge kernel or too few perturbations and compare coefficient stability
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.